In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import asyncio
from hedging_dataset_creator.label_transcript import label_transcript_sentences
import pandas as pd

COMPANY = 'ASML'
transcript_dir = Path(f"data/Transcripts/{COMPANY}")
transcript_files = sorted(transcript_dir.glob("*.txt"))
COOLDOWN_SECONDS = 60

print(f"Found {len(transcript_files)} {COMPANY} transcripts")
for idx, transcript_file in enumerate(transcript_files):
    print(f"Processing {transcript_file.name}")
    await label_transcript_sentences(str(transcript_file))
    if idx < len(transcript_files) - 1:
        print(f"Cooldown: waiting {COOLDOWN_SECONDS} seconds before next transcript...")
        await asyncio.sleep(COOLDOWN_SECONDS)

print(f"Done processing {COMPANY} transcripts")


In [11]:
from speech_parser.transcript_parser import parse_transcript_to_json
from hedging_dataset_creator.sentence_tokenizer import sentence_tokenizer
from hedging_dataset_creator.hedging_labeller import hedging_labeller
TARGET_TRANSCRIPT = Path("data/Transcripts/ASML/2017-Apr-19-ASML.txt")
HALF_COOLDOWN_SECONDS = 30

if not TARGET_TRANSCRIPT.exists():
    raise FileNotFoundError(f"Transcript not found: {TARGET_TRANSCRIPT}")

file_stem = TARGET_TRANSCRIPT.stem
parse_transcript_to_json(str(TARGET_TRANSCRIPT))

processed_json_path = Path("data/processed") / f"{file_stem}.json"
sentences_df = sentence_tokenizer(processed_json_path).reset_index(drop=True)

midpoint = len(sentences_df) // 2
first_half = sentences_df.iloc[:midpoint].copy()
second_half = sentences_df.iloc[midpoint:].copy()

print(f"Total sentences: {len(sentences_df)}")
print(f"First half: {len(first_half)}")
print(f"Second half: {len(second_half)}")

classified_parts = []
if not first_half.empty:
    classified_parts.append(await hedging_labeller(first_half, semaphore=asyncio.Semaphore(10)))

if not first_half.empty and not second_half.empty:
    print(f"Waiting {HALF_COOLDOWN_SECONDS} seconds before labelling second half...")
    await asyncio.sleep(HALF_COOLDOWN_SECONDS)

if not second_half.empty:
    classified_parts.append(await hedging_labeller(second_half, semaphore=asyncio.Semaphore(10)))

classified_sentences = (
    pd.concat(classified_parts, ignore_index=True)
    if classified_parts
    else pd.DataFrame(columns=["sentence", "isHedge"])
)

output_csv_path = Path("data/hedging_dataset/ASML") / f"{file_stem}.csv"
output_csv_path.parent.mkdir(parents=True, exist_ok=True)
classified_sentences.to_csv(output_csv_path, index=False)

print(f"Saved: {output_csv_path}")
display(classified_sentences.head())


Total sentences: 347
First half: 173
Second half: 174


Labelling hedging sentences: 100%|██████████| 18/18 [00:02<00:00,  6.54batch/s]


Waiting 30 seconds before labelling second half...


Labelling hedging sentences: 100%|██████████| 18/18 [00:03<00:00,  4.75batch/s]

Saved: data\hedging_dataset\ASML\2017-Apr-19-ASML.csv


,sentence,isHedge
0,"Thank you, Peter.",0.0
1,"And good afternoon and good morning, ladies an...",0.0
2,"This is Craig DeYoung, Vice President of Inves...",0.0
3,Joining me today from ASML's headquarters in V...,0.0
4,The subject of today's call is ASML's 2017 fir...,0.0


In [12]:
csv_dir = Path(f"data/hedging_dataset/{COMPANY}")
csv_files = sorted(csv_dir.glob("*.csv"))

if not csv_files:
    print(f"No CSV files found in {csv_dir}")
else:
    issues_found = False

    for csv_path in csv_files:
        print(csv_path)
        df = pd.read_csv(csv_path)

        if "isHedge" not in df.columns:
            issues_found = True
            print(f"\n[Missing column] {csv_path.name} has no 'isHedge' column.")
            continue

        # Convert to numeric; non-numeric values become NaN
        numeric_col = pd.to_numeric(df["isHedge"], errors="coerce")

        # Valid values are exactly 0 or 1 (0.0 and 1.0 naturally pass)
        invalid_mask = numeric_col.isna() | ~numeric_col.isin([0, 1])

        if invalid_mask.any():
            issues_found = True
            bad_rows = df.loc[invalid_mask, ["isHedge"]].copy()
            bad_rows["row_index"] = bad_rows.index
            print(f"\n[Invalid values] {csv_path.name}")
            print(bad_rows[["row_index", "isHedge"]].to_string(index=False))

    if not issues_found:
        print("All files passed: every 'isHedge' value is numeric 0/1 (including 0.0/1.0).")

data\hedging_dataset\ASML\2016-Apr-20-ASML.csv
data\hedging_dataset\ASML\2016-Jan-20-ASML.csv
data\hedging_dataset\ASML\2016-Jul-20-ASML.csv
data\hedging_dataset\ASML\2016-Oct-19-ASML.csv
data\hedging_dataset\ASML\2017-Apr-19-ASML.csv
data\hedging_dataset\ASML\2017-Jan-18-ASML.csv
data\hedging_dataset\ASML\2017-Jul-19-ASML.csv
data\hedging_dataset\ASML\2017-Oct-18-ASML.csv
data\hedging_dataset\ASML\2018-Apr-18-ASML.csv
data\hedging_dataset\ASML\2018-Jan-17-ASML.csv
data\hedging_dataset\ASML\2018-Jul-18-ASML.csv
data\hedging_dataset\ASML\2018-Oct-17-ASML.csv
data\hedging_dataset\ASML\2019-Apr-17-ASML.csv
data\hedging_dataset\ASML\2019-Jan-23-ASML.csv
data\hedging_dataset\ASML\2019-Jul-17-ASML.csv
data\hedging_dataset\ASML\2019-Oct-16-ASML.csv
data\hedging_dataset\ASML\2020-Apr-15-ASML.csv
data\hedging_dataset\ASML\2020-Jan-22-ASML.csv
data\hedging_dataset\ASML\2020-Jul-15-ASML.csv
All files passed: every 'isHedge' value is numeric 0/1 (including 0.0/1.0).
